<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/banking77-intent-classification/notebooks/pretrained_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/TheFaridKarami/farid-ai-lab.git

fatal: destination path 'farid-ai-lab' already exists and is not an empty directory.


In [2]:
import torch
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import transformers
import datasets
import sklearn
from pathlib import Path

In [3]:
PROJECT_ROOT = (Path("/content/farid-ai-lab")
 / "projects/nlp/banking77-intent-classification")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

In [4]:
print(PROJECT_ROOT.exists())
print(DATA_DIR.exists())
print(TRAIN_PATH.exists())
print(TEST_PATH.exists())

True
True
True
True


In [5]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

In [6]:
print(train_df.describe())

                                  text                     label
count                            10003                     10003
unique                           10003                        77
top     I am still waiting on my card?  card_payment_fee_charged
freq                                 1                       187


In [7]:
print(train_df.dtypes)

text     object
label    object
dtype: object


In [8]:
print(train_df.head())

                                                text         label
0                     I am still waiting on my card?  card_arrival
1  What can I do if my card still hasn't arrived ...  card_arrival
2  I have been waiting over a week. Is the card s...  card_arrival
3  Can I track my card while it is in the process...  card_arrival
4  How do I know if I will get my card, or if it ...  card_arrival


In [9]:
print(train_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10003 entries, 0 to 10002
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    10003 non-null  object
 1   label   10003 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB
None


In [10]:
print(train_df.isnull().sum())

text     0
label    0
dtype: int64


In [11]:
print(train_df.columns.to_list())

['text', 'label']


In [12]:
print(train_df["label"].nunique())

77


In [13]:
print(train_df.value_counts().describe())

count    10003.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: count, dtype: float64


In [14]:
print(train_df["label"].isna().sum())

0


In [15]:
print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

print("Training classes:", train_df["label"].nunique())
print("Testing classes:", test_df["label"].nunique())

print("\nTraining class distribution:")
print(train_df["label"].value_counts().sort_values())

print("\nClasses missing from test set:")
print(set(train_df["label"]) - set(test_df["label"]))

Training samples: 10003
Testing samples: 3080
Training classes: 77
Testing classes: 77

Training class distribution:
label
contactless_not_working                              35
virtual_card_not_working                             41
card_acceptance                                      59
card_swallowed                                       61
lost_or_stolen_card                                  82
                                                   ... 
cash_withdrawal_charge                              177
wrong_amount_of_cash_received                       180
balance_not_updated_after_cheque_or_cash_deposit    181
direct_debit_payment_not_recognised                 182
card_payment_fee_charged                            187
Name: count, Length: 77, dtype: int64

Classes missing from test set:
set()


In [16]:
print(len(train_df))
print(len(test_df))
print("--" * 20)
print(train_df["label"].nunique())
print(test_df["label"].nunique())

10003
3080
----------------------------------------
77
77


In [17]:
print(train_df.duplicated().sum())
print("$" * 30)
print(train_df["text"].duplicated().sum())
print("$" * 30)
print(test_df["text"].duplicated().sum())
print("$" * 30)
train_df_overlab = set(train_df["text"])
test_df_overlab  = set(test_df["text"])
print(len(train_df_overlab.intersection(test_df_overlab)))
print(train_df_overlab.intersection(test_df_overlab))

0
$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
0
$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
0
$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
0
set()


In [18]:
print(train_df.groupby("label")["text"].nunique())

label
Refund_not_showing_up                      162
activate_my_card                           159
age_limit                                  110
apple_pay_or_google_pay                    126
atm_support                                 87
                                          ... 
virtual_card_not_working                    41
visa_or_mastercard                         135
why_verify_identity                        121
wrong_amount_of_cash_received              180
wrong_exchange_rate_for_cash_withdrawal    163
Name: text, Length: 77, dtype: int64


In [19]:
from sklearn.model_selection import train_test_split
x = train_df["text"]
y = train_df["label"]

X_train , X_val, y_train , y_val = train_test_split(x, y, random_state = 42, test_size = 0.2, stratify = y)

In [20]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)
test_df_encoded = le.transform(test_df["label"])

In [24]:
label2idx = {label:idx for idx, label in  enumerate(le.classes_)}
idx2label = {idx:label for label, idx in label2idx.items()}

In [29]:
print("Number of classes:", len(le.classes_))
print("First 10 mappings:", list(label2idx.items())[:10])
print("Training labels shape:", y_train_encoded.shape)
print("Validation labels shape:", y_val_encoded.shape)
print("Testing labels shape:", test_df_encoded .shape)

Number of classes: 77
First 10 mappings: [('Refund_not_showing_up', 0), ('activate_my_card', 1), ('age_limit', 2), ('apple_pay_or_google_pay', 3), ('atm_support', 4), ('automatic_top_up', 5), ('balance_not_updated_after_bank_transfer', 6), ('balance_not_updated_after_cheque_or_cash_deposit', 7), ('beneficiary_not_allowed', 8), ('cancel_transfer', 9)]
Training labels shape: (8002,)
Validation labels shape: (2001,)
Testing labels shape: (3080,)


In [34]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"text":X_train.tolist() ,"labels":y_train_encoded.tolist()})
val_dataset = Dataset.from_dict({"text":X_val.tolist(), "labels":y_val_encoded.tolist()})
test_dataset = Dataset.from_dict({"text":test_df["text"], "labels": test_df_encoded.tolist()})

In [48]:
from transformers import AutoTokenizer, pipeline, AutoProcessor

MODEL_NAME = "distilbert/distilbert-base-uncased"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_tokens  = tokenizer(X_train.tolist(), add_special_tokens=True, truncation = False).to(DEVICE)



3
